# Configuration Optimization Maps

This notebook sweeps folding geometry and scan length for three science cases, then visualizes the recovered analytical SNR at the science reference wavelength.

The goal is to expose how `delta_x_m` and `n_steps` trade spectral resolution, folding-order complexity, and recovered sensitivity.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from mkid_ifts_sim import InstrumentConfig, load_template, snr_from_time


candidate_steps = [256, 512, 1024, 2048]
candidate_delta_x = [6.0e-6, 3.0e-6, 2.0e-6, 1.0e-6]
science_cases = {
    "H-alpha emission survey": {
        "source": load_template("hii_region", line_flux=2.0e-2),
        "ref_nm": 656.3,
        "t_total_s": 1800.0,
    },
    "Faint galaxy continuum": {
        "source": load_template("composite_galaxy", magnitude=22.0, band="r"),
        "ref_nm": 750.0,
        "t_total_s": 3600.0,
    },
    "Red-edge OH regime": {
        "source": load_template("composite_galaxy", magnitude=21.0, band="i"),
        "ref_nm": 900.0,
        "t_total_s": 3600.0,
    },
}


def evaluate_case(case: dict[str, object]) -> np.ndarray:
    snr_map = np.zeros((len(candidate_steps), len(candidate_delta_x)))
    for i, n_steps in enumerate(candidate_steps):
        for j, delta_x in enumerate(candidate_delta_x):
            cfg = InstrumentConfig(
                n_steps=n_steps,
                delta_x_m=delta_x,
                t_exp_per_step_s=case["t_total_s"] / n_steps,
                strategy="probabilistic",
                apodization="none",
                moon_phase="new",
            )
            result = snr_from_time(case["source"], cfg, case["t_total_s"])
            snr_map[i, j] = np.interp(case["ref_nm"], result.wavelength_nm[::-1], result.snr[::-1])
    return snr_map


snr_maps = {name: evaluate_case(case) for name, case in science_cases.items()}

In [ ]:
fig, axes = plt.subplots(1, len(snr_maps), figsize=(16, 4.5), constrained_layout=True)

for ax, (name, snr_map) in zip(axes, snr_maps.items()):
    image = ax.imshow(snr_map, aspect="auto", origin="lower", cmap="magma")
    ax.set_title(name)
    ax.set_xlabel("delta_x_m")
    ax.set_ylabel("n_steps")
    ax.set_xticks(range(len(candidate_delta_x)), [f"{value:.1e}" for value in candidate_delta_x], rotation=45)
    ax.set_yticks(range(len(candidate_steps)), [str(value) for value in candidate_steps])
    for (row, col), value in np.ndenumerate(snr_map):
        ax.text(col, row, f"{value:.1f}", ha="center", va="center", color="white", fontsize=8)
    fig.colorbar(image, ax=ax, label="SNR at reference wavelength")

plt.show()

In [ ]:
for name, snr_map in snr_maps.items():
    best_index = np.unravel_index(np.argmax(snr_map), snr_map.shape)
    best_steps = candidate_steps[best_index[0]]
    best_delta_x = candidate_delta_x[best_index[1]]
    best_snr = snr_map[best_index]
    print(f"{name}: best SNR={best_snr:.2f} with n_steps={best_steps} and delta_x_m={best_delta_x:.2e}")